<a href="https://colab.research.google.com/github/Soprano2022/Gen-AI/blob/Agentic-AI/QASI_Agent_Enhanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!!pip install litellm

['Collecting litellm',
 '  Downloading litellm-1.80.11-py3-none-any.whl.metadata (29 kB)',
 'Requirement already satisfied: aiohttp>=3.10 in /usr/local/lib/python3.12/dist-packages (from litellm) (3.13.2)',
 'Requirement already satisfied: click in /usr/local/lib/python3.12/dist-packages (from litellm) (8.3.1)',
 'Collecting fastuuid>=0.13.0 (from litellm)',
 '  Downloading fastuuid-0.14.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.1 kB)',
 'Collecting grpcio<1.68.0,>=1.62.3 (from litellm)',
 '  Downloading grpcio-1.67.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.9 kB)',
 'Requirement already satisfied: httpx>=0.23.0 in /usr/local/lib/python3.12/dist-packages (from litellm) (0.28.1)',
 'Requirement already satisfied: importlib-metadata>=6.8.0 in /usr/local/lib/python3.12/dist-packages (from litellm) (8.7.0)',
 'Requirement already satisfied: jinja2<4.0.0,>=3.1.2 in /usr/local/lib/python3.12/dist-packages (from litellm) (3.1.6)',
 

In [1]:
import os
from google.colab import userdata
api_key=userdata.get("OPEN_API_KEY")
os.environ['OPENAI_API_KEY']=api_key


In [2]:
# LiteLLM lets you use OpenAI-style code to talk to models from OpenAI, Azure OpenAI, Anthropic, Google, Meta, Mistral, and others
# — without changing your code.

from litellm import completion
from typing import List, Dict



def generate_response(messages: List[Dict]) -> str:
    """Call LLM to get response"""
    response = completion(
        model="openai/gpt-4o",
        messages=messages,
        max_tokens=1024
    )
    return response.choices[0].message.content


In [3]:
def extract_code_block(response: str) -> str:
    """Extract code block from response"""
    if not '```' in response:
        return response

    code_block = response.split('```')[1].strip()
    if code_block.startswith("python"):
        code_block = code_block[6:]

    return code_block

In [13]:
def develop_custom_function():
  #  Get user input for function description
   print("\nWhat kind of function would you like to create?")
   print("Example: 'A function that calculates the factorial of a number'")
   print("Your description: ", end='')
   function_description = input().strip()
  #  print(function_description)
   messages = [
      {"role": "system", "content": "You are a Python expert helping to develop a function."}
   ]

   # First prompt - Basic function
   messages.append({
      "role": "user",
      "content": f"Write a Python function that {function_description}. Output the function in a ```python code block```."
   })
   initial_function = generate_response(messages)
  #  print(initial_function)
   # Parse the response to get the function code
   initial_function = extract_code_block(initial_function)
   print("\n=== Initial Function ===")
   print(initial_function)

   # Add assistant's response to conversation
   # Notice that I am purposely causing it to forget its commentary and just see the code so that
   # it appears that is always outputting just code.
   messages.append({"role": "assistant", "content": "\`\`\`python\n\n"+initial_function+"\n\n\`\`\`"})
   # Second prompt - Add documentation
   messages.append({
      "role": "user",
      "content": "Add comprehensive documentation to this function, including description, parameters, "
                 "return value, examples, and edge cases. Output the function in a ```python code block```."
   })
   documented_function = generate_response(messages)
   documented_function = extract_code_block(documented_function)
   print("\n=== Documented Function ===")
   print(documented_function)


   # Add documentation response to conversation
   messages.append({"role": "assistant", "content": "\`\`\`python\n\n"+documented_function+"\n\n\`\`\`"})

   # Third prompt - Add test cases
   messages.append({
      "role": "user",
      "content": "Add unittest test cases for this function, including tests for basic functionality, "
                 "edge cases, error cases, and various input scenarios. Output the code in a \`\`\`python code block\`\`\`."
   })
   test_cases = generate_response(messages)
   # We will likely run into random problems here depending on if it outputs JUST the test cases or the
   # test cases AND the code. This is the type of issue we will learn to work through with agents in the course.
   test_cases = extract_code_block(test_cases)
   print("\n=== Test Cases ===")
   print(test_cases)



   # Generate filename from function description
   filename = function_description.lower()
   filename = ''.join(c for c in filename if c.isalnum() or c.isspace())
   filename = filename.replace(' ', '_')[:30] + '.py'

   # Save final version
   with open(filename, 'w') as f:
      f.write(documented_function + '\n\n' + test_cases)

   return documented_function, test_cases, filename


<>:27: SyntaxWarning: invalid escape sequence '\`'
<>:27: SyntaxWarning: invalid escape sequence '\`'
<>:41: SyntaxWarning: invalid escape sequence '\`'
<>:41: SyntaxWarning: invalid escape sequence '\`'
<>:47: SyntaxWarning: invalid escape sequence '\`'
<>:27: SyntaxWarning: invalid escape sequence '\`'
<>:27: SyntaxWarning: invalid escape sequence '\`'
<>:41: SyntaxWarning: invalid escape sequence '\`'
<>:41: SyntaxWarning: invalid escape sequence '\`'
<>:47: SyntaxWarning: invalid escape sequence '\`'
/tmp/ipython-input-1162765102.py:27: SyntaxWarning: invalid escape sequence '\`'
  messages.append({"role": "assistant", "content": "\`\`\`python\n\n"+initial_function+"\n\n\`\`\`"})
/tmp/ipython-input-1162765102.py:27: SyntaxWarning: invalid escape sequence '\`'
  messages.append({"role": "assistant", "content": "\`\`\`python\n\n"+initial_function+"\n\n\`\`\`"})
/tmp/ipython-input-1162765102.py:41: SyntaxWarning: invalid escape sequence '\`'
  messages.append({"role": "assistant", "co

In [14]:
if __name__ == "__main__":
  function_description, test_cases, filename = develop_custom_function()


What kind of function would you like to create?
Example: 'A function that calculates the factorial of a number'
Your description: A fucntion to create fibonacci series

=== Initial Function ===

def fibonacci_series(n_terms):
    if n_terms <= 0:
        return "Please provide a positive integer for the number of terms."
    
    series = []
    a, b = 0, 1
    
    for _ in range(n_terms):
        series.append(a)
        a, b = b, a + b
    
    return series

# Example usage:
# print(fibonacci_series(10))

=== Documented Function ===

def fibonacci_series(n_terms):
    """
    Generate a Fibonacci series up to the specified number of terms.
    
    The Fibonacci sequence is a series of numbers where the next number 
    is found by adding up the two numbers before it. It starts from 0 and 1.

    Parameters:
    n_terms (int): The number of terms to generate in the Fibonacci series. 
                   Must be a positive integer.

    Returns:
    list: A list containing the Fibon

In [15]:
print(function_description)


def fibonacci_series(n_terms):
    """
    Generate a Fibonacci series up to the specified number of terms.
    
    The Fibonacci sequence is a series of numbers where the next number 
    is found by adding up the two numbers before it. It starts from 0 and 1.

    Parameters:
    n_terms (int): The number of terms to generate in the Fibonacci series. 
                   Must be a positive integer.

    Returns:
    list: A list containing the Fibonacci series up to n_terms terms. 
          If n_terms is less than or equal to 0, returns a message requesting a positive integer.

    Examples:
    >>> fibonacci_series(5)
    [0, 1, 1, 2, 3]

    >>> fibonacci_series(0)
    "Please provide a positive integer for the number of terms."
    
    >>> fibonacci_series(1)
    [0]

    Edge Cases:
    - If n_terms is less than or equal to 0, the function returns an explanatory string. 
    - If n_terms is 1, the function returns a list with a single element [0].
    - The function does not h

In [16]:
print(test_cases)


import unittest

class TestFibonacciSeries(unittest.TestCase):

    def test_basic_functionality(self):
        self.assertEqual(fibonacci_series(5), [0, 1, 1, 2, 3])
        self.assertEqual(fibonacci_series(10), [0, 1, 1, 2, 3, 5, 8, 13, 21, 34])

    def test_edge_cases(self):
        self.assertEqual(fibonacci_series(1), [0])
        self.assertEqual(fibonacci_series(0), "Please provide a positive integer for the number of terms.")
        self.assertEqual(fibonacci_series(-5), "Please provide a positive integer for the number of terms.")

    def test_non_integer_input(self):
        with self.assertRaises(TypeError):
            fibonacci_series("ten")
        with self.assertRaises(TypeError):
            fibonacci_series(5.5)

    def test_large_input(self):
        # Test a reasonably large input, but within practical limits
        result = fibonacci_series(20)
        self.assertEqual(result, [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 418

In [17]:
print(filename)

a_fucntion_to_create_fibonacci.py
